In [1]:
# -*- coding: utf-8 -*-
"""
Created on Wed Aug 13 14:40:21 2025

@author: dayı
"""

# === Gerekli Kütüphaneler ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import optuna
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error, 
                           mean_absolute_percentage_error, explained_variance_score, max_error)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, ARDRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor,
    BaggingRegressor, ExtraTreesRegressor, StackingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.pipeline import make_pipeline
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import ScatterChart, Reference, LineChart
import io
import base64

# === Model Listesi ===
models_list = [
    "LinearRegression", "Ridge", "Lasso", "ElasticNet", "BayesianRidge", "ARDRegression",
    "SVR", "DecisionTree", "RandomForest", "GradientBoosting", "AdaBoost",
    "Bagging", "ExtraTrees", "XGBoost", "LightGBM", "CatBoost", "KNN", "MLP", "Stacking"
]

# === Gerekli Sütunlar ===
selected_cols = ['ET_mm_month','elevation','weather_dewpoint_temp','MSI','AVG_SR_B6_B6',
                 'aspect','slope','PRI','spei','weather_solar_radiation',]

# === Adjusted R² Hesaplama Fonksiyonu ===
def adjusted_r2_score(y_true, y_pred, n_features):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    if n - n_features - 1 <= 0:
        return r2
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

# === Feature Importance Alma Fonksiyonu ===
def get_feature_importance(model, model_name, feature_names):
    """Model tipine göre feature importance değerlerini döndürür"""
    try:
        if hasattr(model, 'feature_importances_'):
            return dict(zip(feature_names, model.feature_importances_))
        elif hasattr(model, 'coef_'):
            # Linear modeller için katsayıların mutlak değerini kullan
            return dict(zip(feature_names, np.abs(model.coef_)))
        elif model_name == "Stacking" and hasattr(model, 'final_estimator_'):
            if hasattr(model.final_estimator_, 'coef_'):
                return dict(zip(feature_names, np.abs(model.final_estimator_.coef_)))
        else:
            # Diğer durumlar için boş dictionary döndür
            return {name: 0 for name in feature_names}
    except:
        return {name: 0 for name in feature_names}

# === Learning Curve Hesaplama Fonksiyonu ===
def calculate_learning_curve(model, X, y, cv=5):
    """Learning curve verilerini hesaplar"""
    try:
        train_sizes, train_scores, val_scores = learning_curve(
            model, X, y, cv=cv, n_jobs=-1, 
            train_sizes=np.linspace(0.1, 1.0, 10),
            random_state=42
        )
        
        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        val_mean = np.mean(val_scores, axis=1)
        val_std = np.std(val_scores, axis=1)
        
        return {
            'train_sizes': train_sizes,
            'train_mean': train_mean,
            'train_std': train_std,
            'val_mean': val_mean,
            'val_std': val_std
        }
    except:
        return None

# === Parametre Arama Fonksiyonu ===
def get_model(trial, model_name):
    if model_name == "LinearRegression":
        return LinearRegression()
    elif model_name == "Ridge":
        return Ridge(alpha=trial.suggest_float("alpha", 1e-3, 100, log=True))
    elif model_name == "Lasso":
        return Lasso(alpha=trial.suggest_float("alpha", 1e-3, 100, log=True))
    elif model_name == "ElasticNet":
        return ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-3, 100, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0)
        )
    elif model_name == "BayesianRidge":
        return BayesianRidge()
    elif model_name == "ARDRegression":
        return ARDRegression()
    elif model_name == "SVR":
        return SVR(
            C=trial.suggest_float("C", 0.1, 100, log=True),
            epsilon=trial.suggest_float("epsilon", 0.01, 1.0),
            kernel=trial.suggest_categorical("kernel", ["linear", "rbf"])
        )
    elif model_name == "DecisionTree":
        return DecisionTreeRegressor(max_depth=trial.suggest_int("max_depth", 3, 20))
    elif model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 3, 20),
            random_state=42
        )
    elif model_name == "GradientBoosting":
        return GradientBoostingRegressor(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            random_state=42
        )
    elif model_name == "AdaBoost":
        return AdaBoostRegressor(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            random_state=42
        )
    elif model_name == "Bagging":
        return BaggingRegressor(n_estimators=trial.suggest_int("n_estimators", 10, 100))
    elif model_name == "ExtraTrees":
        return ExtraTreesRegressor(n_estimators=trial.suggest_int("n_estimators", 50, 200))
    elif model_name == "XGBoost":
        return XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            objective="reg:squarederror", verbosity=0, random_state=42
        )
    elif model_name == "LightGBM":
        return LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            random_state=42, verbose=-1
        )
    elif model_name == "CatBoost":
        return CatBoostRegressor(
            iterations=trial.suggest_int("iterations", 50, 200),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            depth=trial.suggest_int("depth", 3, 10),
            verbose=0,
            random_state=42
        )
    elif model_name == "KNN":
        return KNeighborsRegressor(n_neighbors=trial.suggest_int("n_neighbors", 2, 20))
    elif model_name == "MLP":
        return MLPRegressor(
            hidden_layer_sizes=trial.suggest_categorical("hidden_layer_sizes", [(64,), (128,), (128,64)]),
            alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
            learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True),
            max_iter=500, random_state=42
        )
    elif model_name == "Stacking":
        base_models = [
            ("rf", RandomForestRegressor(n_estimators=50, random_state=42)),
            ("xgb", XGBRegressor(n_estimators=50, random_state=42, verbosity=0)),
            ("ridge", Ridge(alpha=1.0))
        ]
        final_estimator = Ridge()
        return StackingRegressor(estimators=base_models, final_estimator=final_estimator)

# === Ana işlemler ===
folder_path = "/kaggle/input/land-senti"
csv_files = ["v"]  # Tek dosya için liste formatında

results = []
graph_data = {}
feature_importance_data = {}
learning_curve_data = {}
excel_path = "model_4_5_full_kfold_hib_tum_ozellik_2aylık.xlsx"

for file in csv_files:
    print(f"\n✨ Dosya işleniyor: {file}")
    df = pd.read_csv("/kaggle/input/2aylik/ikiAy4_5hib.csv")
    
    drop_cols = [
        'Buday', 'ekim', 'HERBST Toplam', 'FUNGUST Toplam', 'dap', 're',
        '20-20-0', '0.33', '20-32-0', 'region_name', 'date', 
        'weather_wind_speed', 'weather_wind_direction', 'weather_pressure_kpa',
        'weather_evaporation_mm', 'TVI', 'DVI', 'RVI', 'MSAVI', 'NDRE'
    ]
    
    y = df['rekolte_x'].values.astype(np.float32)
    X = df.drop(columns=drop_cols + ['rekolte_x'], errors='ignore')
    df.columns = df.columns.str.strip()
    #################################################################################
    # Seçilen özellikleri kullan
    #feat_cols = [c for c in selected_cols if c in X.columns]
    #X = X[feat_cols]
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for model_name in models_list:
        print(f"\u2699\ufe0f Model: {model_name}")

        def objective(trial):
            scores = []
            for train_idx, val_idx in kf.split(X):
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_val_scaled = scaler.transform(X_val)

                model = get_model(trial, model_name)
                model.fit(X_train_scaled, y_train)
                preds = model.predict(X_val_scaled)
                scores.append(r2_score(y_val, preds))

            return np.mean(scores)

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=25)
        best_params = study.best_params

        # Metrics için listeler
        r2s, maes, rmses, mses, mapes, adj_r2s, evs, max_errors = [], [], [], [], [], [], [], []
        y_real_all, y_pred_all = [], []
        feature_importances = []

        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_val_scaled = scaler.transform(X_val)

            model = get_model(optuna.trial.FixedTrial(best_params), model_name)
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_val_scaled)

            # Tüm metrikleri hesapla
            r2s.append(r2_score(y_val, preds))
            maes.append(mean_absolute_error(y_val, preds))
            mse = mean_squared_error(y_val, preds)
            mses.append(mse)
            rmses.append(np.sqrt(mse))
            
            # MAPE hesaplama (sıfır değerlerden kaçınmak için)
            mask = y_val != 0
            if np.any(mask):
                mapes.append(mean_absolute_percentage_error(y_val[mask], preds[mask]))
            else:
                mapes.append(np.inf)
                
            adj_r2s.append(adjusted_r2_score(y_val, preds, X.shape[1]))
            evs.append(explained_variance_score(y_val, preds))
            max_errors.append(max_error(y_val, preds))
            
            y_real_all.extend(y_val)
            y_pred_all.extend(preds)
            
            # Feature importance
            importance = get_feature_importance(model, model_name, X.columns)
            feature_importances.append(importance)

        # Ortalama feature importance hesapla
        avg_feature_importance = {}
        if feature_importances:
            for feature in X.columns:
                avg_feature_importance[feature] = np.mean([imp.get(feature, 0) for imp in feature_importances])

        # Learning curve hesapla (tüm veri ile)
        scaler_full = StandardScaler()
        X_scaled_full = scaler_full.fit_transform(X)
        model_full = get_model(optuna.trial.FixedTrial(best_params), model_name)
        lc_data = calculate_learning_curve(model_full, X_scaled_full, y)

        # Sonuçları kaydet
        result = {
            "Dosya": file,
            "Model": model_name,
            "R2": np.mean(r2s),
            "MAE": np.mean(maes),
            "RMSE": np.mean(rmses),
            "MSE": np.mean(mses),
            "MAPE": np.mean([m for m in mapes if m != np.inf]) if any(m != np.inf for m in mapes) else np.nan,
            "Adjusted_R2": np.mean(adj_r2s),
            "Explained_Variance": np.mean(evs),
            "Max_Error": np.mean(max_errors),
            "Params": str(best_params)
        }
        results.append(result)

        # Grafik verilerini sakla
        graph_data[f"{file}_{model_name}"] = {
            'y_real': y_real_all,
            'y_pred': y_pred_all,
            'residuals': np.array(y_real_all) - np.array(y_pred_all)
        }
        
        # Feature importance verilerini sakla
        if avg_feature_importance:
            feature_importance_data[f"{file}_{model_name}"] = avg_feature_importance
        
        # Learning curve verilerini sakla
        if lc_data:
            learning_curve_data[f"{file}_{model_name}"] = lc_data

# === Sonuçlar ve Excel Dosyası Oluşturma ===
df_results = pd.DataFrame(results)

# En iyi 4 modeli bul (R2 skoruna göre)
top_4_models = df_results.nlargest(4, 'R2')
print(f"\n🏆 En İyi 4 Model:")
print(top_4_models[['Model', 'R2', 'MAE', 'RMSE']].to_string(index=False))

# Excel dosyası oluştur

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Ana sonuçlar
    df_results.to_excel(writer, sheet_name='Model_Results', index=False)
    
    # En iyi 4 model için grafik verileri
    for idx, (_, model_row) in enumerate(top_4_models.iterrows()):
        model_key = f"{model_row['Dosya']}_{model_row['Model']}"
        
        if model_key in graph_data:
            data = graph_data[model_key]
            
            # Scatter plot verisi
            scatter_df = pd.DataFrame({
                'Gercek_Degerler': data['y_real'],
                'Tahmin_Degerler': data['y_pred']
            })
            scatter_df.to_excel(writer, sheet_name=f'Scatter_{model_row["Model"]}', index=False)
            
            # Residuals verisi
            residuals_df = pd.DataFrame({
                'Residuals': data['residuals'],
                'Tahmin_Degerler': data['y_pred']
            })
            residuals_df.to_excel(writer, sheet_name=f'Residuals_{model_row["Model"]}', index=False)
            
            # Residuals histogram verisi
            hist_data, bin_edges = np.histogram(data['residuals'], bins=30)
            hist_df = pd.DataFrame({
                'Bin_Centers': (bin_edges[:-1] + bin_edges[1:]) / 2,
                'Frequency': hist_data
            })
            hist_df.to_excel(writer, sheet_name=f'Hist_{model_row["Model"]}', index=False)
    
    # Feature Importance verileri
    if feature_importance_data:
        # Tüm modeller için feature importance
        all_importance_data = []
        for model_key, importance_dict in feature_importance_data.items():
            for feature, importance in importance_dict.items():
                file_name, model_name = model_key.split('_', 1)
                all_importance_data.append({
                    'Dosya': file_name,
                    'Model': model_name,
                    'Feature': feature,
                    'Importance': importance
                })
        
        importance_df = pd.DataFrame(all_importance_data)
        importance_df.to_excel(writer, sheet_name='Feature_Importance', index=False)
    
    # Learning Curve verileri
    if learning_curve_data:
        all_lc_data = []
        for model_key, lc_dict in learning_curve_data.items():
            if lc_dict:
                file_name, model_name = model_key.split('_', 1)
                for i, size in enumerate(lc_dict['train_sizes']):
                    all_lc_data.append({
                        'Dosya': file_name,
                        'Model': model_name,
                        'Train_Size': size,
                        'Train_Score_Mean': lc_dict['train_mean'][i],
                        'Train_Score_Std': lc_dict['train_std'][i],
                        'Val_Score_Mean': lc_dict['val_mean'][i],
                        'Val_Score_Std': lc_dict['val_std'][i]
                    })
        
        if all_lc_data:
            lc_df = pd.DataFrame(all_lc_data)
            lc_df.to_excel(writer, sheet_name='Learning_Curves', index=False)

print(f"\n✅ Tüm işlemler tamamlandı. Geliştirilmiş sonuçlar '{excel_path}' olarak kaydedildi.")
print(f"\n📊 Eklenen yeni metrikler: MSE, MAPE, Adjusted R², Explained Variance Score, Max Error")
print(f"📈 Eklenen grafik verileri: Scatter plots, Residuals, Histograms, Feature Importance, Learning Curves")
print(f"\nExcel dosyasında ayrı sayfalar:")
print("- Model_Results: Tüm model sonuçları")
print("- Scatter_[Model]: En iyi 4 model için gerçek vs tahmin verileri")
print("- Residuals_[Model]: En iyi 4 model için hata dağılım verileri")
print("- Hist_[Model]: En iyi 4 model için hata histogram verileri")
print("- Feature_Importance: Tüm modeller için özellik önem verileri")
print("- Learning_Curves: Tüm modeller için öğrenme eğrisi verileri")


✨ Dosya işleniyor: v


[I 2026-11-07 18:44:31,990] A new study created in memory with name: no-name-85271e00-f3f6-48f6-94e4-80aae683d53f
[I 2026-11-07 18:44:32,065] Trial 0 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,094] Trial 1 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,125] Trial 2 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,155] Trial 3 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,185] Trial 4 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.


⚙️ Model: LinearRegression


[I 2026-11-07 18:44:32,216] Trial 5 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,245] Trial 6 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,274] Trial 7 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,303] Trial 8 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,332] Trial 9 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,361] Trial 10 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2026-11-07 18:44:32,390] Trial 11 finished with value: 0.439517376812281 and parameters: {}. Best is trial 0 with value: 0.439517376812281.
[I 2

⚙️ Model: Ridge


[I 2026-11-07 18:44:34,501] Trial 6 finished with value: 0.4382151969931759 and parameters: {'alpha': 0.003968589753725817}. Best is trial 3 with value: 0.4547095845742768.
[I 2026-11-07 18:44:34,528] Trial 7 finished with value: 0.44456313769980244 and parameters: {'alpha': 0.15806260373716713}. Best is trial 3 with value: 0.4547095845742768.
[I 2026-11-07 18:44:34,555] Trial 8 finished with value: 0.4554251173160715 and parameters: {'alpha': 3.2082331468382126}. Best is trial 8 with value: 0.4554251173160715.
[I 2026-11-07 18:44:34,584] Trial 9 finished with value: 0.43832337109549 and parameters: {'alpha': 0.005143091272515736}. Best is trial 8 with value: 0.4554251173160715.
[I 2026-11-07 18:44:34,631] Trial 10 finished with value: 0.45078996178534814 and parameters: {'alpha': 0.6801599778850195}. Best is trial 8 with value: 0.4554251173160715.
[I 2026-11-07 18:44:34,662] Trial 11 finished with value: 0.4555410405207785 and parameters: {'alpha': 3.5741976703250575}. Best is trial 1

... (31 outputs left out)

[I 2026-11-07 18:52:06,293] Trial 0 finished with value: 0.37581570818039695 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 2.7943421110206533e-05, 'learning_rate_init': 0.009448373954407683}. Best is trial 0 with value: 0.37581570818039695.
[I 2026-11-07 18:52:08,980] Trial 1 finished with value: -5.407880389018046 and parameters: {'hidden_layer_sizes': (64,), 'alpha': 0.014262526698075343, 'learning_rate_init': 0.0005281318830021647}. Best is trial 0 with value: 0.37581570818039695.
[I 2026-11-07 18:52:15,049] Trial 2 finished with value: 0.407738809382557 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 1.4233232161226918e-05, 'learning_rate_init': 0.004019929856182682}. Best is trial 2 with value: 0.407738809382557.
[I 2026-11-07 18:52:17,715] Trial 3 finished with value: 0.2949030862653439 and parameters: {'hidden_layer_sizes': (64,), 'alpha': 0.0009678119504172723, 'learning_rate_init': 0.0044344142890840425}. Best is trial 2 with value: 0.407738809382557.
[

⚙️ Model: Stacking


[I 2026-11-07 18:54:29,758] Trial 0 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242793901.
[I 2026-11-07 18:54:37,946] Trial 1 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242793901.
[I 2026-11-07 18:54:46,075] Trial 2 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242793901.
[I 2026-11-07 18:54:54,310] Trial 3 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242793901.
[I 2026-11-07 18:55:02,573] Trial 4 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242793901.
[I 2026-11-07 18:55:10,850] Trial 5 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242793901.
[I 2026-11-07 18:55:19,168] Trial 6 finished with value: 0.5747751242793901 and parameters: {}. Best is trial 0 with value: 0.5747751242


🏆 En İyi 4 Model:
       Model       R2       MAE      RMSE
  ExtraTrees 0.655940 52.596352 70.938471
RandomForest 0.630406 55.161850 73.599817
     Bagging 0.627622 55.062682 73.914986
    CatBoost 0.617523 54.384614 74.644374

✅ Tüm işlemler tamamlandı. Geliştirilmiş sonuçlar 'model_4_5_full_kfold_hib_tum_ozellik_2aylık.xlsx' olarak kaydedildi.

📊 Eklenen yeni metrikler: MSE, MAPE, Adjusted R², Explained Variance Score, Max Error
📈 Eklenen grafik verileri: Scatter plots, Residuals, Histograms, Feature Importance, Learning Curves

Excel dosyasında ayrı sayfalar:
- Model_Results: Tüm model sonuçları
- Scatter_[Model]: En iyi 4 model için gerçek vs tahmin verileri
- Residuals_[Model]: En iyi 4 model için hata dağılım verileri
- Hist_[Model]: En iyi 4 model için hata histogram verileri
- Feature_Importance: Tüm modeller için özellik önem verileri
- Learning_Curves: Tüm modeller için öğrenme eğrisi verileri
